# Agent Memory Toolkit – Demo



This notebook walks through the **Agent Memory Toolkit** library using the synchronous `CosmosMemoryClient` class:



1. **Setup** – Install dependencies and load environment variables

2. **Local memory operations** – `add_local`, `get_local`, `update_local`, `delete_local`

3. **Cosmos DB operations** – `add_cosmos`, `get_memories`

4. **Azure Durable Function – Thread Summary** – `generate_thread_summary()` with `CosmosMemoryClient`

5. **Azure Durable Function – Extract Facts** – `extract_facts()` with `CosmosMemoryClient`

6. **Azure Durable Function – User Summary** – `generate_user_summary()` and `get_user_summary()` with `CosmosMemoryClient`

7. **Vector search** – `search_cosmos()` with `CosmosMemoryClient`

8. **Automatic processing (Change Feed)** – Write turns and let the change feed trigger process them automatically



> **Local hosting:** Sections 4–7 require the Azure Durable Functions host running locally via `func start` (see [local_testing.md](../Docs/local_testing.md) for setup). Section 8 additionally requires the change feed settings configured in `local.settings.json`.

>

> **Cosmos provisioning:** `serverless` is the default throughput mode. If you set `COSMOS_DB_THROUGHPUT_MODE=autoscale`, the toolkit applies the shared `COSMOS_DB_AUTOSCALE_MAX_RU` cap to the `memories`, `counter`, and `leases` containers created by `create_memory_store()`.



> For the **async** API (`AsyncCosmosMemoryClient`), see [Demo_async.ipynb](Demo_async.ipynb).

## 1. Setup

Install/import dependencies and load environment variables from `.env`.

In [ ]:
import os, json

from dotenv import load_dotenv

from azure.identity import DefaultAzureCredential



# Add parent directory to path so we can import the package easily

import sys

sys.path.insert(0, os.path.abspath(".."))



from agent_memory_toolkit import CosmosMemoryClient



# Load environment variables from .env in the repo root

load_dotenv(os.path.join("..", ".env"))



print("COSMOS_DB_ENDPOINT:", os.getenv("COSMOS_DB_ENDPOINT"))

print("COSMOS_DB_DATABASE:", os.getenv("COSMOS_DB_DATABASE"))

print("COSMOS_DB_CONTAINER:", os.getenv("COSMOS_DB_CONTAINER"))

print("COSMOS_DB_COUNTERS_CONTAINER:", os.getenv("COSMOS_DB_COUNTERS_CONTAINER", "counter"))

print("COSMOS_DB_LEASE_CONTAINER:", os.getenv("COSMOS_DB_LEASE_CONTAINER", "leases"))

print("COSMOS_DB_THROUGHPUT_MODE:", os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"))

print("COSMOS_DB_AUTOSCALE_MAX_RU:", os.getenv("COSMOS_DB_AUTOSCALE_MAX_RU", "1000"))

In [ ]:
# Create a CosmosMemoryClient instance

memory = CosmosMemoryClient(

    cosmos_endpoint=os.getenv("COSMOS_DB_ENDPOINT"),

    cosmos_database=os.getenv("COSMOS_DB_DATABASE"),

    cosmos_container=os.getenv("COSMOS_DB_CONTAINER"),

    cosmos_counter_container=os.getenv("COSMOS_DB_COUNTERS_CONTAINER", "counter"),

    cosmos_lease_container=os.getenv("COSMOS_DB_LEASE_CONTAINER", "leases"),

    cosmos_throughput_mode=os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"),

    cosmos_autoscale_max_ru=int(os.getenv("COSMOS_DB_AUTOSCALE_MAX_RU", "1000")),

    ai_foundry_endpoint=os.getenv("AI_FOUNDRY_ENDPOINT"),

    embedding_model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-large"),

    adf_endpoint=os.getenv("ADF_ENDPOINT", "http://localhost:7071/api"),

    adf_key=os.getenv("ADF_KEY", ""),

    use_default_credential=True,

    cosmos_credential=DefaultAzureCredential()

)



print("CosmosMemoryClient instance created")

print("Throughput mode:", os.getenv("COSMOS_DB_THROUGHPUT_MODE", "serverless"))

print("Local memory store:", memory.local_memory)

## 2. Local Memory Operations

### 2a. Add memories with `add_local`

Each memory has a `user_id`, `role`, `content`, optional `type` (raw/summary/fact), and optional `metadata`.

In [ ]:
import uuid
THREAD_ID = str(uuid.uuid4())
print(f"Thread ID: {THREAD_ID}\n")

In [ ]:
# Add sample conversation: weather in Seattle → booking a trip (6 turns)
memory.add_local(
    user_id="user-001", role="user", thread_id=THREAD_ID,
    content="What's the weather like in Seattle this weekend?",
)
memory.add_local(
    user_id="user-001", role="agent", thread_id=THREAD_ID,
    content="This weekend Seattle will be around 55°F with partly cloudy skies on Saturday and light rain expected Sunday.",
)
memory.add_local(
    user_id="user-001", role="user", thread_id=THREAD_ID,
    content="That sounds nice enough. Can you help me book a trip to Seattle for this weekend?",
)
memory.add_local(
    user_id="user-001", role="agent", thread_id=THREAD_ID,
    content="Sure! I found round-trip flights departing Friday evening and returning Sunday night. There are also several hotels in downtown Seattle with availability. Would you like me to look at specific airlines or neighborhoods?",
)
memory.add_local(
    user_id="user-001", role="user", thread_id=THREAD_ID,
    content="Something near Pike Place Market would be great. And keep the flights under $300 round trip if possible.",
)
memory.add_local(
    user_id="user-001", role="agent", thread_id=THREAD_ID,
    content="I found a round-trip on Alaska Airlines for $275 and two hotel options within a 5-minute walk of Pike Place Market: the Inn at the Market ($189/night) and a Hilton Garden Inn ($145/night). Want me to reserve one of these?",
)

print(f"Added {len(memory.local_memory)} memories")
print(json.dumps(memory.get_local(), indent=2))

### 2b. Query memories with `get_local`

Retrieve all memories, or filter by `memory_id`, `user_id`, `role`, or `memory_type`. Filters are combined with AND logic.

In [ ]:
# Get all memories
all_memories = memory.get_local()
print(f"Total memories: {len(all_memories)}\n")

# Filter by user_id
user1_memories = memory.get_local(user_id="user-001")
print(f"Memories for user-001: {len(user1_memories)}")

# Filter by role
tool_memories = memory.get_local(role="tool")
print(f"Tool memories: {len(tool_memories)}")
for m in tool_memories:
    print(f"  [{m['id'][:8]}...] {m['content'][:60]}")

# Filter by type
facts = memory.get_local(memory_type="fact")
print(f"\nFact memories: {len(facts)}")
for m in facts:
    print(f"  [{m['id'][:8]}...] {m['content']}")

# Combine filters: user-001 + agent role
user1_agent = memory.get_local(user_id="user-001", role="agent")
print(f"\nAgent memories for user-001: {len(user1_agent)}")

### 2c. Update a memory with `update_local`

Update any combination of `content`, `role`, `memory_type`, or `metadata` for an existing memory by its `id`.

In [ ]:
# Update the user's budget constraint to be more specific
target_id = memory.local_memory[4]["id"]  # "Something near Pike Place Market..."
print(f"Before update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}\n")

memory.update_local(
    memory_id=target_id,
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)
print(f"After update:\n{json.dumps(memory.get_local(memory_id=target_id)[0], indent=2)}")

### 2d. Delete a memory with `delete_local`

Remove a memory by its `id`.

In [ ]:
# Delete the tool memory (index 2 – the tool call)
tool_memory_id = memory.local_memory[2]["id"]
print(f"Deleting memory {tool_memory_id[:8]}...")
memory.delete_local(tool_memory_id)

# Verify it's gone
print(f"\nRemaining memories: {len(memory.get_local())}")
for m in memory.get_local():
    print(f" [{m['thread_id'][:8]}...]  [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:50]}")

## 3. Cosmos DB Operations

### 3a. Cosmos DB Connection

The client auto-connects to Cosmos DB when `cosmos_endpoint` is provided in the constructor. You can also call `connect_cosmos()` explicitly to reconnect or override connection parameters.

> **Prerequisites:**
> - A Cosmos DB for NoSQL account with a database and container matching your `.env` values
> - The container should have a [vector embedding policy](https://learn.microsoft.com/azure/cosmos-db/nosql/vector-search) configured on the `embedding` field
> - Entra ID / managed identity RBAC role (e.g. *Cosmos DB Built-in Data Contributor*)
> - An Azure AI Foundry embedding model deployment for `add_cosmos` and `search_cosmos`

In [ ]:
# Already connected via constructor — call connect_cosmos() only if you need to reconnect
print(f"Connected: {memory._container_client is not None}")

### 3b. Add memories to Cosmos DB with `add_cosmos`

In [ ]:
memory.push_to_cosmos()

In [ ]:
# Push a new thread directly to Cosmos DB without adding to local memory first
new_thread_id = str(uuid.uuid4())
print(f"New Thread ID: {new_thread_id}\n")

# Add memories directly to Cosmos DB using add_cosmos
memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="Can you recommend some good restaurants in New York City?",
)
memory.add_cosmos(
    user_id="user-002", role="tool", thread_id=new_thread_id,
    content='{"query": "top restaurants NYC", "results": ["Carbone", "Nobu", "Katz\'s Deli", "Le Bernardin"]}',
    metadata={"tool_name": "restaurant_search", "tool_call_id": "call_abc123"},
)
memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="Absolutely! NYC has incredible dining options. For Italian, try Carbone in Greenwich Village. For sushi, Nobu in Tribeca is world-class. For a classic NYC experience, Katz's Delicatessen on the Lower East Side is a must.",
)
memory.add_cosmos(
    user_id="user-002", role="user", thread_id=new_thread_id,
    content="I love Italian food. Are there any options that are budget-friendly?",
)
memory.add_cosmos(
    user_id="user-002", role="agent", thread_id=new_thread_id,
    content="For budget-friendly Italian in NYC, check out L'industrie Pizzeria in Williamsburg or Artichoke Basille's Pizza. Both are highly rated and won't break the bank.",
)

# Verify the memories were added directly to Cosmos DB (not in local memory)
print(f"Local memory count (should be unchanged): {len(memory.local_memory)}\n")

cosmos_results = memory.get_memories(user_id="user-002", thread_id=new_thread_id)
print(f"Memories in Cosmos DB for new thread: {len(cosmos_results)}")
for r in cosmos_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} {r['content'][:60]}")

### 3c. Retrieve memories from Cosmos DB with `get_memories`

Supports the same filters as `get_local`: `memory_id`, `user_id`, `role`, `memory_type`.

In [ ]:
# Get all memories for user-001
results = memory.get_memories(user_id="user-001")
print(f"Memories for user-001: {len(results)}\n")
for r in results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

# Get only agent memories
agent_results = memory.get_memories(role="agent")
print(f"\nAgent memories: {len(agent_results)}")
for r in agent_results:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

### 3d. Update & Delete in Cosmos DB

`update_cosmos` and `delete_cosmos` work just like their local counterparts. If the content changes, the embedding is automatically re-generated.

In [ ]:
# Update the user's budget message to add a hotel budget constraint
user_msgs = memory.get_memories(user_id="user-001", role="user")
target = [m for m in user_msgs if "Pike Place" in m["content"]][0]
print(f"Before: {target['content']}\n")

memory.update_cosmos(
    memory_id=target["id"],
    content="Something near Pike Place Market would be great. Keep flights under $300 round trip and hotels under $200/night.",
    metadata={"edited": True, "reason": "user clarified hotel budget"},
)

updated = memory.get_memories(memory_id=target["id"])[0]
print(f"After:  {updated['content']}")

In [ ]:
# Delete the tool memory from Cosmos
tool_mems = memory.get_memories(user_id="user-002", role="tool")
print(tool_mems[0])
if tool_mems:
    memory.delete_cosmos(
        tool_mems[0]["id"],
        thread_id=tool_mems[0]["thread_id"],
        user_id=tool_mems[0]["user_id"],
    )
    print(f"Deleted tool memory {tool_mems[0]['id'][:8]}...")

# Verify
remaining = memory.get_memories()
print(f"\nRemaining memories in Cosmos DB: {len(remaining)}")
for r in remaining:
    print(f"  [{r['thread_id'][:8]}...] [{r['id'][:8]}...] role={r['role']:<6} type={r['type']:<8} {r['content'][:50]}")

### 3e. Retrieve a full thread with `get_thread`

`get_thread` fetches all memories for a given `thread_id` from Cosmos DB, returned in chronological order (oldest first).

- **`thread_id`** (required) – the thread to retrieve
- **`user_id`** (optional) – filter to a specific user
- **`recent_k`** (optional) – return only the *k* most recent documents; omit to get all

In [ ]:
# Pick a thread_id from an existing memory in Cosmos DB
sample = memory.get_memories(user_id="user-001")
if sample:
    thread_id = sample[0]["thread_id"]
    print(f"Using thread_id: {thread_id}\n")

    # Get all documents in the thread
    thread_all = memory.get_thread(thread_id=thread_id)
    print(f"All memories in thread: {len(thread_all)}")
    for m in thread_all:
        print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:60]}")

    # Get only the 2 most recent documents
    thread_recent = memory.get_thread(thread_id=thread_id, recent_k=2)
    print(f"\nMost recent 2 memories:")
    for m in thread_recent:
        print(f"  [{m['thread_id'][:8]}...] [{m['id'][:8]}...] role={m['role']:<6} type={m['type']:<8} {m['content'][:60]}")

    # Filter by user_id as well
    thread_user = memory.get_thread(thread_id=thread_id, user_id="user-001",)
    print(f"\nThread memories for user-001: {len(thread_user)}")
else:
    print("No memories found in Cosmos DB – run the add_cosmos cells first.")

## 4. Azure Durable Function – Thread Summary\n

The `generate_thread_summary` method on `CosmosMemoryClient` triggers the **Azure Durable Functions** orchestrator to:\n
1. Query Cosmos DB for memories matching a `user_id` + `thread_id`
2. Optionally keep only the **k most recent** documents (`recent_k`)
3. Send the content and metadata to an AI Foundry LLM for summarisation
4. Generate a vector embedding of the summary
5. Store the resulting summary back into Cosmos DB as a memory of type `"summary"`

### Setup – start the Durable Function locally

Before running the code below, you need the Functions host running in a separate terminal:

```bash
# 1. Start Azurite (Azure Storage emulator) – required by the Functions runtime
azurite --silent --location /tmp/azurite --debug /tmp/azurite/debug.log

# 2. Log in to Azure (needed for DefaultAzureCredential in the activities)
az login

# 3. Install dependencies and start the Functions host
cd azure_functions
pip install -r azure_functions/requirements.txt
func start
```

You should see the registered functions listed in the terminal output:
`memory_orchestrator`, `load_memories`, `generate_embeddings`, `store_results`, `generate_thread_summary`, `http_start`.\n

### Function key (required)

The HTTP trigger uses **`AuthLevel.FUNCTION`** — a function key must be included with every request.

- **Locally:** Run `func keys list` in the `azure_functions/` directory to get the default key after `func start`.
- **Deployed:** Go to **Azure Portal → Function App → App keys** and copy the **default** key, or use:
  ```bash
  az functionapp keys list --name <app> --resource-group <rg> --query "functionKeys.default" -o tsv
  ```

Set the key in your `.env` file as `ADF_KEY=<your-key>`. The `CosmosMemoryClient` class passes it automatically via the `adf_key` parameter.

> **Note:** Make sure `azure_functions/local.settings.json` has your real Cosmos DB and AI Foundry endpoint values (copy them from your `.env` file).

In [ ]:
# Verify ADF endpoint is configured (set in the Setup cell above)
print(f"ADF endpoint: {memory.adf_endpoint}")
print(f"ADF key set:  {bool(memory.adf_key)}")

In [ ]:
# Pick a thread_id + user_id from existing Cosmos data
sample = memory.get_memories(user_id="user-001")
if not sample:
    raise RuntimeError("No memories for user-001 in Cosmos – run the add_cosmos cells first.")

thread_id = sample[0]["thread_id"]
user_id = sample[0]["user_id"]
print(f"Summarizing thread_id={thread_id}  user_id={user_id}\n")

# Call summarize (all memories in the thread)
result = memory.generate_thread_summary(user_id=user_id, thread_id=thread_id)

In [ ]:
print("\nSummary document:")
output = result.get("output", {})
print(f"  [{output.get('thread_id', '')[:8]}...] user={output.get('user_id', ''):<10} type={output.get('type', ''):<8} {output.get('content', '')[:100]}")

len(output['embedding'])

## 5. Azure Durable Function – Extract Facts

`extract_facts` triggers the same orchestrator but calls the **extract_facts** activity instead of generate_thread_summary. The LLM extracts discrete factual statements from the thread and stores them in Cosmos DB as a memory with `type: \"fact\"`.\n

> **Prerequisites** are the same as section 4 — the Functions host must be running.

In [ ]:
# Extract facts from the same thread
facts_result = memory.extract_facts(user_id=user_id, thread_id=thread_id)

print("Status:", facts_result.get("runtimeStatus"))

In [ ]:
output = facts_result.get("output", {})
print(f"\nExtracted {len(output)} fact(s):\n")
for fact in output:
    print(f"  [{fact.get('thread_id', '')[:8]}...] user={fact.get('user_id', ''):<10} type={fact.get('type', ''):<8} {fact.get('content', '')[:80]}")

## 6. Azure Durable Function – User Summary

`generate_user_summary` triggers the same orchestrator but calls the **generate_user_summary** activity. It aggregates memories **across all threads** for a user and produces a structured profile covering preferences, account state, compliance details, and behavioural patterns. The result is stored in Cosmos DB as a memory with `type: "user_summary"`.

- **`user_id`** (required) – the user to build the profile for
- **`thread_ids`** (optional) – limit to specific threads; omit for all threads
- **`recent_k`** (optional) – per-thread recency limit

Retrieve the current user summary at any time with `get_user_summary(user_id)` — useful for priming new conversations with user context.

> **Prerequisites** are the same as sections 4 and 5 — the Functions host must be running.

In [ ]:
# Generate a user summary across all threads for user-001
user_summary_result = memory.generate_user_summary(user_id=user_id)

print("Status:", user_summary_result.get("runtimeStatus"))
print("\nUser Summary document:")
print(json.dumps(user_summary_result.get("output", {}), indent=2))

In [ ]:
# Retrieve the stored user summary from Cosmos DB
stored_summary = memory.get_user_summary(user_id=user_id)
if stored_summary:
    print("User Summary for", user_id)
    print(stored_summary[0]["content"])
else:
    print("No user summary found — run the generate_user_summary cell first.")

### 7. Vector search with `search_cosmos`

`search_cosmos` embeds a natural-language query via your AI Foundry model and runs a `VectorDistance` similarity search in Cosmos DB. Optionally filter by `user_id` or `thread_id`.


In [ ]:
results = memory.search_cosmos(
    search_terms="What did the user ask about the weather?",
    user_id="user-001",
    top_k=3,
)

In [ ]:
print(f"Top {len(results)} results:\n")
for r in results:
    score = r.get("score", "N/A")
    print(f"  [{r['id'][:8]}...] score={score}  {r['content'][:60]}")

### 8. Automatic Processing (Change Feed)



The toolkit includes a Cosmos DB change feed trigger that automatically fires thread summaries, fact extraction, and user summaries when configurable message count thresholds are crossed.



**Prerequisites:**

- The Azure Functions host must be running (`func start`)

- `local.settings.json` must include change feed settings:

  - `COSMOS_DB__accountEndpoint` pointing to your Cosmos account

  - `COSMOS_DB_COUNTERS_CONTAINER` set to `"counter"`

  - `COSMOS_DB_LEASE_CONTAINER` set to `"leases"`

  - At least one threshold > 0 (e.g. `THREAD_SUMMARY_EVERY_N=3`)

- `create_memory_store()` must have provisioned the `counter` and `leases` containers in the same database



The cells below write enough turns to cross the threshold, then poll for auto-generated derived memories.

In [ ]:
# Write turns to trigger automatic processing
# Assumes THREAD_SUMMARY_EVERY_N=3 in local.settings.json
import uuid, time

changefeed_thread = str(uuid.uuid4())
print(f"Thread ID: {changefeed_thread}")

for i in range(3):
    memory.add_cosmos(
        user_id="user-001",
        thread_id=changefeed_thread,
        role="user",
        content=f"Change feed test message {i+1}: Tell me about vector databases.",
    )
    print(f"  Wrote turn {i+1}")

print("\nTurns written. The change feed trigger should fire within a few seconds.")
print("Waiting 15 seconds for processing...")
time.sleep(15)

In [ ]:
# Check for auto-generated derived memories
results = memory.get_memories(user_id="user-001", thread_id=changefeed_thread, memory_type="summary")
if results:
    print("Auto-generated thread summary found!")
    for r in results:
        print(f"  [{r['id'][:20]}...] {r['content'][:100]}")
else:
    print("No summary yet — the change feed may need more time. Try re-running this cell after a few seconds.")

print()

facts = memory.get_memories(user_id="user-001", thread_id=changefeed_thread, memory_type="fact")
if facts:
    print(f"Auto-extracted facts ({len(facts)}):")
    for f in facts:
        print(f"  • {f['content'][:80]}")
else:
    print("No facts yet — check FACT_EXTRACTION_EVERY_N in local.settings.json.")